# Week 3 Day 2 Lab 2 - The Gateway and the Policy

## Deploying UnderCover behind an AgentCore Gateway

In this lab we deploy the UnderCover app to AWS Lambda, so it has an address on the internet, and put an AgentCore Gateway in front of it.

The Gateway turns the claim agent's routes into MCP tools and sends the app's API key with every call. A Cedar Policy on the Gateway refuses any payment over $1,000 before the call reaches the app. On Day 3 the agent moves onto these tools. Today we call them from Python.

## 4 parts:

PART 1: The UnderCover app on Lambda

PART 2: The AgentCore Gateway

PART 3: The Policy

PART 4: Teardown

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">You need the AWS account and the <code>undercover</code> IAM user from Day 1, and today's build of the product, yours or <code>week3day2-end</code>, with <code>UNDERCOVER_API_KEY</code> in your <code>.env</code> file.<br/><br/>
            The app on Lambda is inside AWS's free allowance. The Gateway, the Policy and the secret that holds the API key cost a fraction of a cent while they exist, and Part 4 deletes them. The app stays on Lambda for Day 3.<br/><br/>
            If you're on the free route, <a href="week3day2_lab2free.ipynb">week3day2_lab2free.ipynb</a> deploys the app on Render instead. The Gateway and the Policy need AWS, so for Parts 2 and 3 you can watch me.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Watching your AWS spend</h2>
            <span style="color:#118AB2;">Before you start, open a terminal in the <code>undercover</code> folder and run <code>uv run monitor.py</code>. It opens a page in your browser with what your AWS account has spent today and over the last 30 days, with and without your credits. It reads the figures from two daily budgets with no alerts, which it creates in your account the first time it runs. Budgets like these are free. AWS updates the figures up to three times a day.<br/><br/>
            For the official source of truth, please review Billing and Cost Management in your AWS console. Always have budgets with billing alerts, like the one from Day 1.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import asyncio
import json
import os
import shutil
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import boto3
import httpx
import requests
from botocore.auth import SigV4Auth as BotocoreSigV4Auth
from botocore.awsrequest import AWSRequest
from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.shared.exceptions import McpError

## Connect to AWS

boto3 uses the access key that `aws configure` saved on Day 1, in us-east-1.

In [ ]:
load_dotenv(override=True)

session = boto3.Session()
region = session.region_name
identity = session.client("sts").get_caller_identity()
account = identity["Account"]

print(f"Working as {identity['Arn']} in {region}")

In [ ]:
lambda_client = session.client("lambda")
iam = session.client("iam")
logs = session.client("logs")
agentcore = session.client("bedrock-agentcore-control")
cloudwatch = session.client("cloudwatch")

Everything we create is named `undercover...` and tagged `project=undercover`, so the teardown can find it by name.

In [ ]:
FUNCTION_NAME = "undercover-app"
APP_ROLE_NAME = "undercover-app-role"
LOG_GROUP = f"/aws/lambda/{FUNCTION_NAME}"
KEY_PROVIDER_NAME = "undercover-app-key"
GATEWAY_ROLE_NAME = "undercover-gateway-role"
GATEWAY_NAME = "undercover-gateway"
TARGET_NAME = "undercover-app"
ENGINE_NAME = "undercover_policy_engine"
PROJECT_TAG = {"project": "undercover"}

The deployed app reads the same Supabase database as your laptop, and needs the agent's API key.

If you built the product yourself, your coding agent added `UNDERCOVER_API_KEY` to your `.env`. If you're using my build on the `week3day2-end` branch, add it yourself. Letters and numbers are safest, and this prints a random one in a terminal:

```
uv run python -c "import secrets; print(secrets.token_hex(32))"
```

Then run the cells again from Connect to AWS.

In [ ]:
for name in ["DATABASE_URL", "VITE_SUPABASE_URL", "VITE_SUPABASE_PUBLISHABLE_KEY", "UNDERCOVER_API_KEY"]:
    print(f"{name}: {'found' if os.getenv(name) else 'missing, please add it to .env'}")

API_KEY = os.environ["UNDERCOVER_API_KEY"]

## Part 1: The UnderCover app on Lambda

### Building the dashboard

`npm run build` builds the dashboard into `dashboard/dist`, with the Supabase lines from `.env`. The app serves these files beside the API, so the dashboard and the API share one address.

In [ ]:
!cd .. && npm run build

### The package

Lambda runs our code from a zip file. The packages the backend needs on a server are listed in the `server` group in `pyproject.toml`. `uv export` writes them out at the exact versions in `uv.lock`, and `uv pip install` fetches the Linux versions, since Lambda runs on Linux.

In [ ]:
BUILD = Path("../build")
PACKAGE = BUILD / "lambda"
ZIP = BUILD / "undercover-app.zip"

shutil.rmtree(PACKAGE, ignore_errors=True)
PACKAGE.mkdir(parents=True)

!uv export --only-group server --no-hashes --no-emit-project --quiet --output-file {BUILD / "requirements.txt"}
!uv pip install --quiet --target {PACKAGE} --python-platform x86_64-manylinux2014 --python-version 3.12 --only-binary :all: -r {BUILD / "requirements.txt"}

Then our own code: the backend, the data it was seeded from, and the dashboard's build.

In [ ]:
for folder in ["backend", "data"]:
    shutil.copytree(Path("..") / folder, PACKAGE / folder, ignore=shutil.ignore_patterns("tests", "__pycache__"))
shutil.copytree(Path("../dashboard/dist"), PACKAGE / "dashboard" / "dist")

shutil.make_archive(str(ZIP.with_suffix("")), "zip", PACKAGE)
print(f"{ZIP.name}: {ZIP.stat().st_size / 1_000_000:.1f} MB (Lambda takes up to 50 MB)")

### The log group and the role

We create the log group ourselves, with a retention period, so Lambda doesn't make one that keeps logs forever. The app's role lets Lambda run it and write to that log group, and nothing else.

In [ ]:
try:
    logs.create_log_group(logGroupName=LOG_GROUP)
    print(f"Created log group {LOG_GROUP}")
except logs.exceptions.ResourceAlreadyExistsException:
    print(f"Log group {LOG_GROUP} already exists")

logs.put_retention_policy(logGroupName=LOG_GROUP, retentionInDays=7)
print("Logs are kept for 7 days")

In [ ]:
lambda_trust = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}],
}

try:
    app_role = iam.create_role(
        RoleName=APP_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(lambda_trust),
        Description="The role the UnderCover app runs as on Lambda",
        Tags=[{"Key": key, "Value": value} for key, value in PROJECT_TAG.items()],
    )["Role"]
    print(f"Created role {APP_ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    app_role = iam.get_role(RoleName=APP_ROLE_NAME)["Role"]
    print(f"Role {APP_ROLE_NAME} already exists")

logging_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": f"arn:aws:logs:{region}:{account}:log-group:{LOG_GROUP}:*",
        }
    ],
}
iam.put_role_policy(RoleName=APP_ROLE_NAME, PolicyName="undercover-app-logging", PolicyDocument=json.dumps(logging_policy))
print("The role can write to the log group")

### The function

AWS starts a copy of the function for each request that arrives while the others are busy, and each copy handles one request at a time. So on Lambda the backend opens a database connection for each request, through Supabase's transaction pooler on port 6543. Keeping connections open would soon use up the 15 that Supabase's free session pooler allows, with the backend and the agent on your laptop already using some.

In [ ]:
environment = {
    "Variables": {
        "DATABASE_URL": os.environ["DATABASE_URL"].replace(":5432/", ":6543/"),
        "VITE_SUPABASE_URL": os.environ["VITE_SUPABASE_URL"],
        "UNDERCOVER_API_KEY": API_KEY,
    }
}

The first time, the new role can take a few seconds to reach Lambda, so `create_function` tries again. If the function already exists, we update its code and settings in place, which keeps its address.

In [ ]:
settings = {
    "Runtime": "python3.12",
    "Handler": "backend.main.handler",
    "Timeout": 30,
    "MemorySize": 1024,
    "Environment": environment,
}


def create_function(zip_bytes):
    """Create the function, waiting for a new role to reach Lambda."""
    for attempt in range(10):
        try:
            return lambda_client.create_function(
                FunctionName=FUNCTION_NAME, Role=app_role["Arn"], Code={"ZipFile": zip_bytes}, Tags=PROJECT_TAG, **settings
            )
        except lambda_client.exceptions.InvalidParameterValueException:
            print("Waiting for the new role to reach Lambda")
            time.sleep(3)
    raise RuntimeError("Lambda could not use the role")


zip_bytes = ZIP.read_bytes()

try:
    lambda_client.get_function(FunctionName=FUNCTION_NAME)
    lambda_client.update_function_code(FunctionName=FUNCTION_NAME, ZipFile=zip_bytes)
    lambda_client.get_waiter("function_updated_v2").wait(FunctionName=FUNCTION_NAME)
    lambda_client.update_function_configuration(FunctionName=FUNCTION_NAME, Role=app_role["Arn"], **settings)
    lambda_client.get_waiter("function_updated_v2").wait(FunctionName=FUNCTION_NAME)
    print(f"Updated function {FUNCTION_NAME}")
except lambda_client.exceptions.ResourceNotFoundException:
    create_function(zip_bytes)
    lambda_client.get_waiter("function_active_v2").wait(FunctionName=FUNCTION_NAME)
    print(f"Created function {FUNCTION_NAME}")

### The Function URL

A Function URL gives the function a public HTTPS address at no charge. Anyone can reach it, which is why the agent's routes need the API key and the handlers sign in. It needs two permissions, or every request gets a 403.

In [ ]:
try:
    url_config = lambda_client.create_function_url_config(FunctionName=FUNCTION_NAME, AuthType="NONE")
except lambda_client.exceptions.ResourceConflictException:
    url_config = lambda_client.get_function_url_config(FunctionName=FUNCTION_NAME)

public_access = [
    {"StatementId": "public-url", "Action": "lambda:InvokeFunctionUrl", "FunctionUrlAuthType": "NONE"},
    {"StatementId": "public-invoke", "Action": "lambda:InvokeFunction", "InvokedViaFunctionUrl": True},
]

for statement in public_access:
    try:
        lambda_client.add_permission(FunctionName=FUNCTION_NAME, Principal="*", **statement)
    except lambda_client.exceptions.ResourceConflictException:
        pass

app_url = url_config["FunctionUrl"].rstrip("/")
print(app_url)

### Trying it

The first request takes a few seconds while Lambda starts a copy of the app. After that, requests take a fraction of a second.

In [ ]:
for attempt in range(2):
    started = time.perf_counter()
    response = requests.get(f"{app_url}/api/health", timeout=60)
    print(f"{response.status_code} {response.json()} in {time.perf_counter() - started:.1f} seconds")

The agent's routes need the API key, and Lambda has it in its environment.

In [ ]:
claim_url = f"{app_url}/api/claims/CLM-1901"

print("Without the key:", requests.get(claim_url, timeout=30).status_code)
print("With the key:", requests.get(claim_url, headers={"X-API-Key": API_KEY}, timeout=30).status_code)

### The deployed dashboard

Open the address above in your browser and sign in as Dana Whitfield. You'll see the same claims as on your laptop, since both read the same Supabase database.

The agent's service is still on your laptop, where the deployed app can't reach it until Day 4. So for now, dispatches, evaluations and rounds from the deployed dashboard say that the agent's service isn't running. A sign-off works, and pays the claim.

## Part 2: The AgentCore Gateway

### The API key in AgentCore Identity

The Gateway sends the app's API key with every call. It keeps the key in AgentCore Identity as an API key credential provider, which stores it in AWS Secrets Manager. Running this cell again updates the key.

In [ ]:
def find_key_providers():
    """The API key credential providers named undercover..."""
    pages = agentcore.get_paginator("list_api_key_credential_providers").paginate()
    return [p["name"] for page in pages for p in page["credentialProviders"] if p["name"].startswith("undercover")]


if KEY_PROVIDER_NAME in find_key_providers():
    agentcore.update_api_key_credential_provider(name=KEY_PROVIDER_NAME, apiKey=API_KEY)
    print(f"Credential provider {KEY_PROVIDER_NAME} already exists, key updated")
else:
    agentcore.create_api_key_credential_provider(name=KEY_PROVIDER_NAME, apiKey=API_KEY, tags=PROJECT_TAG)
    print(f"Created credential provider {KEY_PROVIDER_NAME}")

provider = agentcore.get_api_key_credential_provider(name=KEY_PROVIDER_NAME)
provider_arn = provider["credentialProviderArn"]
secret_arn = provider["apiKeySecretArn"]["secretArn"]

### The Gateway's role

The Gateway runs as a role of its own. Only gateways named `undercover-gateway` in your account can use it. For now it can only fetch the API key, and in Part 3 it gets permission to use the Policy.

In [ ]:
gateway_trust = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": account},
                "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{region}:{account}:gateway/{GATEWAY_NAME}-*"},
            },
        }
    ],
}

try:
    gateway_role = iam.create_role(
        RoleName=GATEWAY_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(gateway_trust),
        Description="The role the UnderCover gateway runs as",
        Tags=[{"Key": key, "Value": value} for key, value in PROJECT_TAG.items()],
    )["Role"]
    print(f"Created role {GATEWAY_ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    gateway_role = iam.get_role(RoleName=GATEWAY_ROLE_NAME)["Role"]
    print(f"Role {GATEWAY_ROLE_NAME} already exists")

To fetch the key, the Gateway goes through its own workload identity and the token vault where AgentCore Identity keeps credentials, so the role needs both as well as the provider and the secret.

In [ ]:
workload_directory = f"arn:aws:bedrock-agentcore:{region}:{account}:workload-identity-directory/default"
workload_identity = [workload_directory, f"{workload_directory}/workload-identity/{GATEWAY_NAME}-*"]
token_vault = provider_arn.split("/apikeycredentialprovider/")[0]

api_key_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": "bedrock-agentcore:GetWorkloadAccessToken", "Resource": workload_identity},
        {"Effect": "Allow", "Action": "bedrock-agentcore:GetResourceApiKey", "Resource": [provider_arn, token_vault, *workload_identity]},
        {"Effect": "Allow", "Action": "secretsmanager:GetSecretValue", "Resource": secret_arn},
    ],
}

iam.put_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName="undercover-gateway-api-key", PolicyDocument=json.dumps(api_key_policy))
print("The role can fetch the app's API key")

### The Gateway

The Gateway speaks MCP, and callers sign their requests with their AWS credentials (`AWS_IAM`), so only users and roles in your account that are allowed to invoke it can use it. `DEBUG` makes its error messages say what went wrong.

In [ ]:
def wait_for(get_resource, label):
    """Wait while an AgentCore resource is being created or updated, then print its status."""
    while True:
        resource = get_resource()
        if resource["status"] not in ("CREATING", "UPDATING", "SYNCHRONIZING"):
            break
        time.sleep(3)
    print(f"{label}: {resource['status']}")
    for reason in resource.get("statusReasons", []):
        print(f"  {reason}")
    return resource


def find_gateways():
    """The gateways named undercover-gateway."""
    pages = agentcore.get_paginator("list_gateways").paginate()
    return [g for page in pages for g in page["items"] if g["name"].startswith(GATEWAY_NAME)]

In [ ]:
def create_gateway():
    """Create the Gateway, waiting for its new role to reach AgentCore."""
    for attempt in range(10):
        try:
            return agentcore.create_gateway(
                name=GATEWAY_NAME,
                description="UnderCover's systems as tools for the claim agent",
                roleArn=gateway_role["Arn"],
                protocolType="MCP",
                authorizerType="AWS_IAM",
                exceptionLevel="DEBUG",
                tags=PROJECT_TAG,
            )
        except agentcore.exceptions.ValidationException as error:
            if "role" not in str(error).lower():
                raise
            print("Waiting for the new role to reach AgentCore")
            time.sleep(3)
    raise RuntimeError("AgentCore could not use the role")


existing = find_gateways()
gateway_id = existing[0]["gatewayId"] if existing else create_gateway()["gatewayId"]

gateway = wait_for(lambda: agentcore.get_gateway(gatewayIdentifier=gateway_id), f"Gateway {gateway_id}")
gateway_arn = gateway["gatewayArn"]
gateway_url = gateway["gatewayUrl"]
print(gateway_url)

### The agent's own API document

The app serves an OpenAPI document with only the eight routes the claim agent uses. Each route becomes a tool, named by its `operation_id`, with its docstring as the description. Signing off, the dashboard's routes and the evaluations aren't in it, so the agent can't call them through the Gateway. FastAPI doesn't add the app's address to the document, so we add it.

In [ ]:
agent_api = requests.get(f"{app_url}/api/agent/openapi.json", headers={"X-API-Key": API_KEY}, timeout=30).json()
agent_api["servers"] = [{"url": app_url}]

for path, operations in agent_api["paths"].items():
    for method, operation in operations.items():
        print(f"{method.upper():<6}{operation['operationId']:<20}{path}")

### The target

The app becomes the Gateway's target, with the document inline and the API key sent in the `X-API-Key` header. Running this cell again updates the target with a fresh document.

In [ ]:
def find_targets(gateway_id):
    """The Gateway's targets."""
    pages = agentcore.get_paginator("list_gateway_targets").paginate(gatewayIdentifier=gateway_id)
    return [t for page in pages for t in page["items"]]


target_settings = {
    "gatewayIdentifier": gateway_id,
    "name": TARGET_NAME,
    "description": "The UnderCover app's routes for the claim agent",
    "targetConfiguration": {"mcp": {"openApiSchema": {"inlinePayload": json.dumps(agent_api)}}},
    "credentialProviderConfigurations": [
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {
                    "providerArn": provider_arn,
                    "credentialParameterName": "X-API-Key",
                    "credentialLocation": "HEADER",
                }
            },
        }
    ],
}

existing = [t for t in find_targets(gateway_id) if t["name"] == TARGET_NAME]

if existing:
    target_id = existing[0]["targetId"]
    agentcore.update_gateway_target(targetId=target_id, **target_settings)
else:
    target_id = agentcore.create_gateway_target(**target_settings)["targetId"]

target = wait_for(lambda: agentcore.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id), f"Target {target_id}")

### Signing our calls

Each request to the Gateway is signed with AWS Signature Version 4, using your IAM user's credentials, as boto3 signs its own calls. This class signs the HTTP requests that the MCP client sends.

In [ ]:
class SigV4Auth(httpx.Auth):
    """Sign each request with AWS Signature Version 4, as boto3 does for its own calls."""

    requires_request_body = True

    def __init__(self, session, service="bedrock-agentcore"):
        self.credentials = session.get_credentials()
        self.region = session.region_name
        self.service = service

    def auth_flow(self, request):
        headers = {"Host": request.url.netloc.decode(), "Content-Type": request.headers.get("content-type", "")}
        aws_request = AWSRequest(method=request.method, url=str(request.url), data=request.content, headers=headers)
        BotocoreSigV4Auth(self.credentials.get_frozen_credentials(), self.service, self.region).add_auth(aws_request)
        for name in ("Authorization", "X-Amz-Date", "X-Amz-Security-Token"):
            if name in aws_request.headers:
                request.headers[name] = aws_request.headers[name]
        yield request

### The tools

The same MCP client as Week 2 Day 2, connecting over HTTP this time. On Day 3 the claim agent gets its tools this way.

In [ ]:
client = MultiServerMCPClient(
    {"undercover": {"transport": "streamable_http", "url": gateway_url, "auth": SigV4Auth(session)}}
)

tools = {tool.name: tool for tool in await client.get_tools()}

for tool in tools.values():
    print(f"{tool.name:<36}{tool.description}")

`call_tool` calls one of the tools and prints what comes back. An error from the app comes back as the tool's result, for the agent to read. The Gateway's own refusal is an MCP error.

In [ ]:
async def call_tool(operation, **arguments):
    """Call one of the app's routes through the Gateway and print what comes back."""
    try:
        result = await tools[f"{TARGET_NAME}___{operation}"].ainvoke(arguments)
        print(result[0]["text"][:300])
    except McpError as error:
        print(f"Refused by the Gateway: {error}")

A new role can take up to a minute to reach AgentCore, so the first calls may fail with "Failed to obtain execution role credentials". This cell tries again until they work.

In [ ]:
for attempt in range(20):
    result = await tools[f"{TARGET_NAME}___get_claim"].ainvoke({"claim_id": "CLM-1901"})
    if "execution role" not in result[0]["text"]:
        break
    print("Waiting for the Gateway's role to reach AgentCore")
    await asyncio.sleep(5)

print(result[0]["text"][:300])

In [ ]:
await call_tool("search_documents", query="unattended")

In [ ]:
await call_tool("get_claim", claim_id="CLM-9999")

## Part 3: The Policy

### The policy engine

A policy engine holds Cedar policies, and the Gateway asks it about every tool call. The Gateway's role needs permission to use it.

In [ ]:
def find_policy_engines():
    """The policy engines named undercover..."""
    pages = agentcore.get_paginator("list_policy_engines").paginate()
    return [e for page in pages for e in page["policyEngines"] if e["name"].startswith("undercover")]


existing = [e for e in find_policy_engines() if e["name"] == ENGINE_NAME]

if existing:
    engine_id = existing[0]["policyEngineId"]
else:
    engine_id = agentcore.create_policy_engine(name=ENGINE_NAME, description="Rules for the UnderCover gateway", tags=PROJECT_TAG)["policyEngineId"]

engine = wait_for(lambda: agentcore.get_policy_engine(policyEngineId=engine_id), f"Policy engine {engine_id}")
engine_arn = engine["policyEngineArn"]

In [ ]:
policy_engine_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": "bedrock-agentcore:GetPolicyEngine", "Resource": engine_arn},
        {
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:AuthorizeAction", "bedrock-agentcore:PartiallyAuthorizeActions"],
            "Resource": [engine_arn, gateway_arn],
        },
    ],
}

iam.put_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName="undercover-gateway-policy-engine", PolicyDocument=json.dumps(policy_engine_policy))
print("The role can use the policy engine")

### Attaching it in log-only mode

In `LOG_ONLY` mode the engine records what it would refuse and lets every call through. That's how you'd try a new rule on real traffic before enforcing it. `update_gateway` needs every setting sent again, and the role's new permission can take a few seconds to arrive.

In [ ]:
def attach_policy_engine(mode):
    """Attach the policy engine to the Gateway in LOG_ONLY or ENFORCE mode."""
    for attempt in range(20):
        try:
            agentcore.update_gateway(
                gatewayIdentifier=gateway_id,
                name=GATEWAY_NAME,
                description=gateway["description"],
                roleArn=gateway_role["Arn"],
                protocolType="MCP",
                authorizerType="AWS_IAM",
                exceptionLevel="DEBUG",
                policyEngineConfiguration={"arn": engine_arn, "mode": mode},
            )
            break
        except agentcore.exceptions.ValidationException as error:
            if "GetPolicyEngine" not in str(error):
                raise
            print("Waiting for the role's new permission to reach AgentCore")
            time.sleep(5)
    updated = wait_for(lambda: agentcore.get_gateway(gatewayIdentifier=gateway_id), f"Gateway {gateway_id}")
    print(f"Policy engine mode: {updated['policyEngineConfiguration']['mode']}")


attach_policy_engine("LOG_ONLY")

### The rules in Cedar

With an engine attached, a call is refused unless a policy permits it, so the first policy permits every tool on our target. The second forbids `pay_claim` with an amount over $1,000. A forbid always wins over a permit.

The amount is a JSON number, which Cedar reads as a `decimal`, so the comparison is `greaterThan` rather than `>`.

In [ ]:
policies = {
    "undercover_allow_agent_tools": f"""permit(
  principal is AgentCore::IamEntity,
  action in AgentCore::Action::"{TARGET_NAME}",
  resource == AgentCore::Gateway::"{gateway_arn}"
);""",
    "undercover_payment_limit": f"""forbid(
  principal is AgentCore::IamEntity,
  action == AgentCore::Action::"{TARGET_NAME}___pay_claim",
  resource == AgentCore::Gateway::"{gateway_arn}"
)
when {{
  context.input.amount.greaterThan(decimal("1000.00"))
}};""",
}

print(policies["undercover_payment_limit"])

In [ ]:
def find_policies(engine_id):
    """The policies in a policy engine."""
    pages = agentcore.get_paginator("list_policies").paginate(policyEngineId=engine_id)
    return [p for page in pages for p in page["policies"]]


existing = {p["name"]: p for p in find_policies(engine_id)}

for name, statement in policies.items():
    if name in existing:
        policy_id = existing[name]["policyId"]
    else:
        policy_id = agentcore.create_policy(policyEngineId=engine_id, name=name, definition={"cedar": {"statement": statement}})["policyId"]
    wait_for(lambda: agentcore.get_policy(policyEngineId=engine_id, policyId=policy_id), f"Policy {name}")

### Which layer says no

Suppose the injected note from Lab 1 had fooled the agent, and it tried to pay the $4,800 on CLM-1901. In log-only mode the Gateway lets the call through and the app's own limit refuses it.

In [ ]:
await call_tool("pay_claim", claim_id="CLM-1901", amount=4800.0)

A payment within the limit passes the Policy too, and the app refuses it for its own reason: CLM-1901 was never approved.

In [ ]:
await call_tool("pay_claim", claim_id="CLM-1901", amount=400.0)

### Enforcing the Policy

Now the Gateway refuses the $4,800 itself, and the call never reaches the app. The $400 call still goes through to the app.

In [ ]:
attach_policy_engine("ENFORCE")

In [ ]:
await call_tool("pay_claim", claim_id="CLM-1901", amount=4800.0)

In [ ]:
await call_tool("pay_claim", claim_id="CLM-1901", amount=400.0)

### The engine's decisions

The policy engine counts its decisions in CloudWatch, a minute or two behind, so run this cell again in a couple of minutes if it shows zeros.

In [ ]:
def decisions(metric, mode):
    """The engine's decisions of one kind in one mode over the last three hours."""
    now = datetime.now(timezone.utc)
    stats = cloudwatch.get_metric_statistics(
        Namespace="AWS/Bedrock-AgentCore",
        MetricName=metric,
        Dimensions=[
            {"Name": "OperationName", "Value": "AuthorizeAction"},
            {"Name": "Mode", "Value": mode},
            {"Name": "PolicyEngine", "Value": engine_id},
        ],
        StartTime=now - timedelta(hours=3),
        EndTime=now,
        Period=3600,
        Statistics=["Sum"],
    )
    return int(sum(point["Sum"] for point in stats["Datapoints"]))


for mode in ["LOG_ONLY", "ENFORCE"]:
    print(f"{mode:<10}allowed {decisions('AllowDecisions', mode)}, refused {decisions('DenyDecisions', mode)}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Three layers</h2>
            <span style="color:#118AB2;">The model check from Lab 1 stops most injected instructions before the agent acts on them. The Policy refuses a payment over $1,000 whatever the agent decides, and the app has its own limit behind that. A handler's sign-off is the only way a payment over $1,000 gets paid, and the sign-off isn't a tool on the Gateway.
            </span>
        </td>
    </tr>
</table>

## Part 4: Teardown

The app stays on Lambda for Day 3, and costs nothing while nobody is using it. We delete the AgentCore pieces, which Day 3 creates again, so nothing is left that bills by the month.

In [ ]:
def find_roles():
    """The IAM roles named undercover..."""
    pages = iam.get_paginator("list_roles").paginate()
    return [r["RoleName"] for page in pages for r in page["Roles"] if r["RoleName"].startswith("undercover")]


def show_resources():
    """Print what today's lab created that still exists."""
    lambda_pages = lambda_client.get_paginator("list_functions").paginate()
    found = {
        "Lambda functions": [f["FunctionName"] for page in lambda_pages for f in page["Functions"] if f["FunctionName"].startswith("undercover")],
        "Gateways": [g["name"] for g in find_gateways()],
        "Policy engines": [e["name"] for e in find_policy_engines()],
        "API key providers": find_key_providers(),
        "IAM roles": find_roles(),
    }
    for label, names in found.items():
        print(f"{label + ':':<20}{', '.join(names) or 'none'}")


show_resources()

Targets go before their gateway and policies before their engine. AgentCore deletes the secret that held the API key a few seconds after the provider.

In [ ]:
for summary in find_gateways():
    for target_summary in find_targets(summary["gatewayId"]):
        agentcore.delete_gateway_target(gatewayIdentifier=summary["gatewayId"], targetId=target_summary["targetId"])
        print(f"Deleted target {target_summary['name']}")
    while find_targets(summary["gatewayId"]):
        time.sleep(3)
    agentcore.delete_gateway(gatewayIdentifier=summary["gatewayId"])
    print(f"Deleted gateway {summary['name']}")

for summary in find_policy_engines():
    for policy_summary in find_policies(summary["policyEngineId"]):
        agentcore.delete_policy(policyEngineId=summary["policyEngineId"], policyId=policy_summary["policyId"])
        print(f"Deleted policy {policy_summary['name']}")
    while find_policies(summary["policyEngineId"]):
        time.sleep(3)
    agentcore.delete_policy_engine(policyEngineId=summary["policyEngineId"])
    print(f"Deleted policy engine {summary['name']}")

for name in find_key_providers():
    agentcore.delete_api_key_credential_provider(name=name)
    print(f"Deleted credential provider {name}")

if GATEWAY_ROLE_NAME in find_roles():
    for policy_name in iam.list_role_policies(RoleName=GATEWAY_ROLE_NAME)["PolicyNames"]:
        iam.delete_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName=policy_name)
    iam.delete_role(RoleName=GATEWAY_ROLE_NAME)
    print(f"Deleted role {GATEWAY_ROLE_NAME}")

If you're stopping the course here, set `DELETE_THE_APP` to `True` and run the cell to delete the app from Lambda as well, with its role and log group. Deleting the function also deletes its address.

In [ ]:
DELETE_THE_APP = False

if DELETE_THE_APP:
    lambda_client.delete_function(FunctionName=FUNCTION_NAME)
    print(f"Deleted function {FUNCTION_NAME}")
    for policy_name in iam.list_role_policies(RoleName=APP_ROLE_NAME)["PolicyNames"]:
        iam.delete_role_policy(RoleName=APP_ROLE_NAME, PolicyName=policy_name)
    iam.delete_role(RoleName=APP_ROLE_NAME)
    print(f"Deleted role {APP_ROLE_NAME}")
    logs.delete_log_group(logGroupName=LOG_GROUP)
    print(f"Deleted log group {LOG_GROUP}")

In [ ]:
time.sleep(10)
show_resources()

Over the next day or two, keep an eye on your spend with `uv run monitor.py`. For the official figures, review Billing and Cost Management in your AWS console.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">A gateway gives a client's security team one place to see and control what every agent can do, with rules they can read and review without reading the agent's code. Trying a new rule in log-only mode on real traffic, before enforcing it, is how you'd introduce one without breaking the agents that depend on the gateway.<br/><br/>
            Consider which rules a gateway would need to enforce for an agent in your own organization.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: add a third Cedar policy that refuses <code>record_decision</code> when the decision is an approval with a payout over $5,000, and try it in log-only mode first.<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>